In [1]:
!pwd

/home/jovyan/work


In [2]:
!ls

'1. Introduccion.ipynb'			    consumer_spark_demo_mysql.py
'2. DataFrame.ipynb'			    emulador_customers.py
'2. ProcesamientoParalelomnn_final.ipynb'   emulador_datos.py
'3. Practica Mysql Hdfs Spark.ipynb'	    emulador_fx2.py
'4. Emulador Stream.ipynb'		    emulador_retail_orders_stream.py
'5 Streaming.ipynb'			    kafka_producer_orders_stream.py
'6 Batch Propuesto.ipynb'		    mysql_kafka_producer.py
 Untitled.ipynb				    productor_fx2.py
 __pycache__				    spark-warehouse
 consumer_spark_demo.py			    spark_mysql.log


In [3]:
!python emulador_datos.py

Clientes cargados: 12435
Productos cargados: 1345
Ultimo order_id: 69670
Ultimo order_item_id: 174612

Simulador iniciado...

Para la tabla orders Orden 69671 | Fecha 2026-08-12 04:17:46.223188 | Cliente 12111 | Estado COMPLETE
Para la tabla order_items Orden_item 174617 | Order_id 69671 | Producto_id 504 | Subtotal $89.97| Total $29.99
Para la tabla orders Orden 69672 | Fecha 2026-08-12 04:17:52.392214 | Cliente 9729 | Estado PROCESSING
Para la tabla order_items Orden_item 174618 | Order_id 69672 | Producto_id 932 | Subtotal $89.97| Total $29.99
Para la tabla orders Orden 69673 | Fecha 2026-08-12 04:18:00.405604 | Cliente 7571 | Estado CLOSED
Para la tabla order_items Orden_item 174621 | Order_id 69673 | Producto_id 505 | Subtotal $90.00| Total $90.00
Para la tabla orders Orden 69674 | Fecha 2026-08-12 04:18:06.418701 | Cliente 7579 | Estado PROCESSING
Para la tabla order_items Orden_item 174625 | Order_id 69674 | Producto_id 860 | Subtotal $1799.97| Total $599.99
Para la tabla orders

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LambdaBatch")
    .config("spark.jars", "/opt/spark/jars/mysql-connector-j-8.4.0.jar")
    .getOrCreate()
)

mysql_url = "jdbc:mysql://mysql:3306/retail_db"

properties = {
    "user": "root",
    "password": "root",
    "driver": "com.mysql.cj.jdbc.Driver"
}

# =====================================================
# 1. INGESTA MYSQL -> RAW
# =====================================================

orders = (
    spark.read
    .jdbc(
        url=mysql_url,
        table="orders",
        properties=properties
    )
)

order_items = (
    spark.read
    .jdbc(
        url=mysql_url,
        table="order_items",
        properties=properties
    )
)

orders.write.mode("overwrite").parquet(
    "hdfs://namenode:8020/lambda/raw/batch/orders"
)

order_items.write.mode("overwrite").parquet(
    "hdfs://namenode:8020/lambda/raw/batch/order_items"
)

/usr/local/lib/python3.8/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
26/08/12 04:26:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
                                                                                

In [5]:
# =====================================================
# 2. RAW -> CLEANSED
# =====================================================

orders_raw = (
    spark.read
    .parquet("hdfs://namenode:8020/lambda/raw/batch/orders")
)

items_raw = (
    spark.read
    .parquet("hdfs://namenode:8020/lambda/raw/batch/order_items")
)

In [6]:
from pyspark.sql.functions import col, trim

orders_cleansed = (
    orders_raw
    .filter(col("order_id").isNotNull())
    .withColumn("order_status", trim(col("order_status")))
)

items_cleansed = (
    items_raw
    .filter(col("order_item_order_id").isNotNull())
    .filter(col("order_item_quantity") > 0)
)

In [7]:
orders_cleansed.write.mode("overwrite").parquet(
    "hdfs://namenode:8020/lambda/cleansed/batch/orders"
)

items_cleansed.write.mode("overwrite").parquet(
    "hdfs://namenode:8020/lambda/cleansed/batch/order_items"
)

In [8]:
# =====================================================
# 3. JOIN
# =====================================================

orders_clean = (
    spark.read
    .parquet("hdfs://namenode:8020/lambda/cleansed/batch/orders")
)

items_clean = (
    spark.read
    .parquet("hdfs://namenode:8020/lambda/cleansed/batch/order_items")
)

ventas_batch = (
    orders_clean
    .join(
        items_clean,
        orders_clean.order_id ==
        items_clean.order_item_order_id,
        "inner"
    )
)

In [9]:
ventas_batch.write.mode("overwrite").parquet(
    "hdfs://namenode:8020/lambda/cleansed/batch/ventas"
)

In [10]:
import time
from pyspark.sql.functions import col, trim

RAW_PATH = "hdfs://namenode:8020/lambda/raw/batch"
CLEANSED_PATH = "hdfs://namenode:8020/lambda/cleansed/batch"

INTERVALO_RAW = 600      # 10 minutos
ciclo = 0

while True:

    ciclo += 1
    print("================================")
    print(f"INICIANDO RAW - ciclo {ciclo}")
    print("================================")

    orders = spark.read.jdbc(url=mysql_url, table="orders", properties=properties)
    order_items = spark.read.jdbc(url=mysql_url, table="order_items", properties=properties)

    orders.write.mode("overwrite").parquet(f"{RAW_PATH}/orders")
    order_items.write.mode("overwrite").parquet(f"{RAW_PATH}/order_items")

    print("RAW FINALIZADO")

    if ciclo % 2 == 0:
        print("--------------------------------")
        print("INICIANDO CLEANSED + JOIN")
        print("--------------------------------")

        orders_clean = (
            orders.filter(col("order_id").isNotNull())
            .withColumn("order_status", trim(col("order_status")))
        )
        items_clean = (
            order_items.filter(col("order_item_order_id").isNotNull())
            .filter(col("order_item_quantity") > 0)
        )

        orders_clean.write.mode("overwrite").parquet(f"{CLEANSED_PATH}/orders")
        items_clean.write.mode("overwrite").parquet(f"{CLEANSED_PATH}/order_items")

        ventas = orders_clean.join(
            items_clean,
            orders_clean.order_id == items_clean.order_item_order_id,
            "inner"
        )
        ventas.write.mode("overwrite").parquet(f"{CLEANSED_PATH}/ventas")

        print("CLEANSED + JOIN FINALIZADO")

    print(f"Esperando {INTERVALO_RAW // 60} minutos...\n")
    time.sleep(INTERVALO_RAW)


INICIANDO RAW - ciclo 1


RAW FINALIZADO
Esperando 10 minutos...

INICIANDO RAW - ciclo 2


RAW FINALIZADO
--------------------------------
INICIANDO CLEANSED + JOIN
--------------------------------


CLEANSED + JOIN FINALIZADO
Esperando 10 minutos...

INICIANDO RAW - ciclo 3


RAW FINALIZADO
Esperando 10 minutos...

INICIANDO RAW - ciclo 4


RAW FINALIZADO
--------------------------------
INICIANDO CLEANSED + JOIN
--------------------------------


26/08/12 04:59:13 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-b0f6cfa9-430a-4e8b-be44-4c1d224fe3e3/09/temp_shuffle_e94fcd31-507c-4d94-b691-9679cb430d8b
26/08/12 04:59:13 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-b0f6cfa9-430a-4e8b-be44-4c1d224fe3e3/16/temp_shuffle_cd991f27-53b5-4742-abe7-f37828a00394
26/08/12 04:59:14 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-b0f6cfa9-430a-4e8b-be44-4c1d224fe3e3/19/temp_shuffle_4313120e-dc4b-481f-abb5-887e5aa6b9f9
26/08/12 04:59:14 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-b0f6cfa9-430a-4e8b-be44-4c1d224fe3e3/19/temp_shuffle_3e7f7133-6dee-4cb4-a45c-0c1eac06cd70
26/08/12 04:59:14 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-b0f6cfa9-430a-4e8b-be44-4c1d224fe3e3/19/temp_shuffle_310dc776-3022-4a41-b9a1-34499ea66c90
26/08/12 04:59:14 WARN DiskBlockObjectWriter: Error deleting /tmp/blockmgr-b0f6cfa9-430a-4e8b-be44-4c1d224fe3e3/35/temp_shuffle_88a4f478-78ca-4a1e-819c-71f6da06c955
26/08/12 0

ConnectionRefusedError: [Errno 111] Connection refused

```
                 MYSQL
                   │
        ┌──────────┴──────────┐
        │                     │
        ▼                     ▼
     BATCH                 KAFKA
   cada 10 min           cada evento
        │                     │
        ▼                     ▼
      RAW                STREAMING
        │                  2 min
        ▼                     │
   CLEANSED                   ▼
        │                   SPEED
        ▼
       JOIN
        │
        ▼
      BATCH
```


```
MySQL
  │
  │ nuevos registros
  ▼
RAW
  │
  ▼
CLEANSED
  │
  ▼
JOIN
  │
  ▼
BATCH
```
